# 🚀 Fluxara Neural Engine Training
Transitioning from CPU to GPU-accelerated training on Google Colab.

In [1]:
# 1. Setup Environment
!pip install torch torchvision numpy scipy tqdm xarray
from google.colab import drive
drive.mount('/content/drive')


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# 2. Copy Data from Drive
# Assume training_data.zip is in Drive root
!cp /content/drive/MyDrive/training_data.zip /content/
!unzip -q /content/training_data.zip -d /content/

In [ ]:
# 3. Import Fluxara Model
import sys
sys.path.append('/content/Fluxara') # Assuming repo is also copied or cloned

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import os
from tqdm.auto import tqdm

class FluxaraDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = data_dir
        self.files = [f for f in os.listdir(data_dir) if f.endswith('.npz')]
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        data = np.load(os.path.join(self.data_dir, self.files[idx]))
        return torch.from_numpy(data['wind_field']), torch.from_numpy(data['footprint'])

In [ ]:
# 4. Training Loop with Live Monitoring
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on {device}")

model = FluxaraInversionNet().to(device)
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=80)

train_loader = DataLoader(FluxaraDataset('/content/training_data'), batch_size=32, shuffle=True)

for epoch in range(80):
    model.train()
    total_loss = 0
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch}"):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        # Physics penalty
        physics_penalty = 0.1 * torch.abs(pred.sum(dim=(1,2,3)) - 1.0).mean()
        (loss + physics_penalty).backward()
        optimizer.step()
        total_loss += loss.item()
    
    scheduler.step()
    print(f"Epoch {epoch} | Loss: {total_loss/len(train_loader):.6f}")
    
    # Save Best
    torch.save(model.state_dict(), 'fluxara_best.pt')